<a href="https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakshiiikashyap/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** Refresh / Content Opportunity Scoring.

**Question:** Can observable content and search-performance signals identify pages that are more likely to be declining and therefore worth review?

**Method choice:** I use three supervised classification models from the toolkit — Logistic Regression, a shallow Decision Tree, and Random Forest. The Decision Tree is easy to interpret, Logistic Regression gives a simple linear baseline, and Random Forest can capture non-linear interactions without relying on complexity alone. I select the model using **Precision@50**, because the practical goal is to rank a small review queue and make the top recommendations useful.

The target is `is_declining_label = 1` when `trend_direction == "down"`. I exclude `trend_direction` and `trend_pct` from the features because they directly define or encode the outcome and would leak the answer into the model.


In [6]:
from pathlib import Path
import zipfile
import pandas as pd

# 1. Find the uploaded ZIP
zip_path = Path("/content/flyrank-ml-internship-main.zip")

# 2. Extract it
extract_path = Path("/content/flyrank_project")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted!")

# 3. Find the CSV anywhere inside the extracted folder
matches = list(extract_path.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found inside the ZIP."
    )

DATA_PATH = matches[0]

# 4. Project root = folder containing data/
PROJECT_ROOT = DATA_PATH.parents[2]

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

# 5. Load data
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.head())

ZIP extracted!
Project root: /content/flyrank_project/flyrank-ml-internship-main
Data path: /content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 44)
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     comme

In [7]:
from pathlib import Path

data_path = Path(
    "/content/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv"
)

print(data_path.exists())
print(data_path)

True
/content/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv


In [10]:
# Setup: extract ZIP and load the anonymized starter data.

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# ZIP uploaded in Colab
ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")

# Extract ZIP
EXTRACT_PATH = Path("/content/flyrank_project")

if not EXTRACT_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

print("ZIP extracted successfully.")

# Find the CSV anywhere inside the extracted project
matches = list(EXTRACT_PATH.rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found inside the ZIP."
    )

DATA_PATH = matches[0]

# Project root = folder containing data/raw
PROJECT_ROOT = DATA_PATH.parents[2]

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

# Load data
df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")

# Week-4 baseline recreated as a deterministic score
df["baseline_score"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["ctr"] < 0.5)
).astype(float)

FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

missing = [
    c for c in FEATURES + ["client_id", "content_id"]
    if c not in df.columns
]

if missing:
    raise KeyError(f"Required columns missing from dataset: {missing}")

print("Model features:", FEATURES)

ZIP extracted successfully.
Project root: /content/flyrank_project/flyrank-ml-internship-main
Data path: /content/flyrank_project/flyrank-ml-internship-main/data/raw/content_refresh_anonymized.csv
Loaded 30,000 rows and 45 columns
Declining rate: 0.542
Model features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']


## 2. Split design

I use a **client-holdout split** when the data contains enough distinct clients. Twenty percent of clients are held out for testing, so pages from a held-out client cannot appear in training. This is more conservative than randomly splitting rows because related pages from the same client can otherwise make performance look artificially strong.

If the client-holdout cannot produce both target classes in train and test, the notebook falls back to a stratified 80/20 row split. The split is fixed with `random_state=42`.

The Week-4 baseline is scored on the **same test rows** and evaluated with the same ranking metric, Precision@50.


In [11]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
all_idx = np.arange(len(df))

client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

split_strategy = "stratified_row_holdout"

if len(unique_clients) >= 5:
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = rng.permutation(unique_clients)
    n_test_clients = max(1, int(round(len(shuffled_clients) * 0.20)))
    test_clients = set(shuffled_clients[:n_test_clients])

    test_mask = client_series.isin(test_clients).to_numpy()
    train_idx = all_idx[~test_mask]
    test_idx = all_idx[test_mask]

    if (
        len(train_idx) > 0
        and len(test_idx) > 0
        and df["is_declining_label"].iloc[train_idx].nunique() == 2
        and df["is_declining_label"].iloc[test_idx].nunique() == 2
    ):
        split_strategy = "client_holdout"

if split_strategy != "client_holdout":
    train_idx, test_idx = train_test_split(
        all_idx,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=df["is_declining_label"],
    )

train_idx = np.asarray(train_idx)
test_idx = np.asarray(test_idx)

print("Split strategy:", split_strategy)
print(f"Train rows: {len(train_idx):,}")
print(f"Test rows:  {len(test_idx):,}")
print(f"Train declining rate: {df['is_declining_label'].iloc[train_idx].mean():.3f}")
print(f"Test declining rate:  {df['is_declining_label'].iloc[test_idx].mean():.3f}")

if split_strategy == "client_holdout":
    overlap = set(client_series.iloc[train_idx]) & set(client_series.iloc[test_idx])
    print("Client overlap:", len(overlap))
    assert len(overlap) == 0, "Client leakage detected!"


Split strategy: client_holdout
Train rows: 27,675
Test rows:  2,325
Train declining rate: 0.555
Test declining rate:  0.391
Client overlap: 0


## 3. Train + compare vs my baseline

The baseline and every model are evaluated on the same held-out test set.

**Primary metric: Precision@50.** This answers: among the 50 highest-ranked pages, what fraction are actually declining?

I also report ROC-AUC, average precision, precision, recall, and F1 so the result is not reduced to one number.


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).copy()
y = df["is_declining_label"].astype(int)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

models = {
    "Logistic Regression": Pipeline([
        ("prep", numeric_pipeline),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Decision Tree": Pipeline([
        ("prep", numeric_pipeline),
        ("model", DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": Pipeline([
        ("prep", numeric_pipeline),
        ("model", RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_true[order].mean())

def evaluate_scores(y_true, scores):
    preds = (scores >= 0.5).astype(int)
    return {
        "Precision@50": precision_at_k(y_true, scores, 50),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(y_true, scores),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "Accuracy": accuracy_score(y_true, preds),
    }

results = []

# Week-4 baseline on exactly the same test set.
baseline_scores = df["baseline_score"].iloc[test_idx].to_numpy()
baseline_metrics = evaluate_scores(y_test, baseline_scores)
results.append({"Method": "Week-4 baseline", **baseline_metrics})

fitted_models = {}
test_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model
    test_scores[name] = scores
    results.append({"Method": name, **evaluate_scores(y_test, scores)})

comparison = (
    pd.DataFrame(results)
    .sort_values("Precision@50", ascending=False)
    .reset_index(drop=True)
)

display(comparison.round(3))

best_model_name = comparison.loc[
    comparison["Method"] != "Week-4 baseline", "Method"
].iloc[0]

best_model_score = test_scores[best_model_name]
baseline_p50 = comparison.loc[
    comparison["Method"] == "Week-4 baseline", "Precision@50"
].iloc[0]
best_p50 = comparison.loc[
    comparison["Method"] == best_model_name, "Precision@50"
].iloc[0]

print(f"Best ML model by Precision@50: {best_model_name}")
print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Best-model Precision@50: {best_p50:.3f}")
print(f"Difference: {best_p50 - baseline_p50:+.3f}")


,Method,Precision@50,ROC-AUC,Average Precision,Precision,Recall,F1,Accuracy
0,Decision Tree,0.64,0.733,0.576,0.577,0.592,0.584,0.671
1,Week-4 baseline,0.60,0.572,0.439,0.544,0.310,0.395,0.629
2,Random Forest,0.38,0.723,0.537,0.568,0.624,0.595,0.668
3,Logistic Regression,0.22,0.664,0.528,0.541,0.704,0.612,0.651


Best ML model by Precision@50: Decision Tree
Baseline Precision@50: 0.600
Best-model Precision@50: 0.640
Difference: +0.040


## 4. Errors and interpretation

The important question is not only whether the model scores higher. I inspect the false positives and false negatives and identify which observable signals the selected model relies on.

I also check that no label-derived feature (`trend_direction` or `trend_pct`) entered the model.


In [13]:
# Error analysis on the held-out test set.
error_frame = df.iloc[test_idx][
    ["content_id", "client_id", "is_declining_label"] + FEATURES
].copy()

error_frame["model_score"] = best_model_score
error_frame["predicted_label"] = (error_frame["model_score"] >= 0.5).astype(int)

false_positives = error_frame[
    (error_frame["predicted_label"] == 1)
    & (error_frame["is_declining_label"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = error_frame[
    (error_frame["predicted_label"] == 0)
    & (error_frame["is_declining_label"] == 1)
].sort_values("model_score", ascending=True)

print(f"False positives: {len(false_positives):,}")
print(f"False negatives: {len(false_negatives):,}")

print("\nTop false positives:")
display(false_positives.head(5))

print("\nTop false negatives:")
display(false_negatives.head(5))

# Feature importance for interpretation.
best_model = fitted_models[best_model_name]
classifier = best_model.named_steps["model"]

if hasattr(classifier, "feature_importances_"):
    importance = pd.Series(
        classifier.feature_importances_,
        index=FEATURES,
        name="importance",
    ).sort_values(ascending=False)
else:
    importance = pd.Series(
        np.abs(classifier.coef_[0]),
        index=FEATURES,
        name="importance",
    ).sort_values(ascending=False)

print("\nFeature importance / absolute coefficient magnitude:")
display(importance.to_frame().round(4))

forbidden = {"trend_direction", "trend_pct"}
used_features = set(FEATURES)
assert not used_features & forbidden, "Leakage feature included!"

print("\nLeakage check: PASSED — trend_direction and trend_pct were not model features.")

print("\nInterpretation:")
print(
    f"The selected model was {best_model_name}. "
    f"Its Precision@50 was {best_p50:.3f} versus "
    f"{baseline_p50:.3f} for the Week-4 baseline on the same held-out rows. "
    "The error review shows that some high-scoring pages are not declining "
    "(false positives), while some declining pages receive lower scores "
    "(false negatives). The result is therefore treated as directional "
    "decision-support rather than proof that a refresh will cause improvement."
)


False positives: 395
False negatives: 371

Top false positives:


,content_id,client_id,is_declining_label,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,model_score,predicted_label
29372,content_56b0e78884ab,client_f74efabef1,0,112,8,148,44.8,0.0,2530.0,0.723178,1
27662,content_9fe06fb9369d,client_f74efabef1,0,175,20,588,8.8,0.0,2519.0,0.723178,1
2471,content_52f107b1f722,client_f74efabef1,0,175,20,90,17.2,0.0,3019.0,0.723178,1
2420,content_37669b78bfe3,client_f74efabef1,0,175,20,378,23.7,0.0,3526.0,0.723178,1
2380,content_96da95476e63,client_f74efabef1,0,125,20,784,7.4,0.0,2598.0,0.723178,1



Top false negatives:


,content_id,client_id,is_declining_label,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,model_score,predicted_label
26859,content_f1762127797f,client_98a3ab7c34,1,91,1,4,0.8,0.00,2826.0,0.0,0
3879,content_34b14c00f80c,client_d4735e3a26,1,308,20,3,0.0,0.00,659.0,0.0,0
27177,content_79ac977c6e0b,client_f74efabef1,1,104,8,3,0.7,0.00,2304.0,0.0,0
22991,content_472ce7ae14c0,client_d4735e3a26,1,300,20,3,0.3,33.33,684.0,0.0,0
5770,content_28b4223f4e5f,client_98a3ab7c34,1,91,1,1,0.0,0.00,3109.0,0.0,0



Feature importance / absolute coefficient magnitude:


,importance
impressions_90d,0.4845
content_age_days,0.2572
avg_position,0.1095
ctr,0.0932
word_count,0.0286
days_since_last_update,0.0271



Leakage check: PASSED — trend_direction and trend_pct were not model features.

Interpretation:
The selected model was Decision Tree. Its Precision@50 was 0.640 versus 0.600 for the Week-4 baseline on the same held-out rows. The error review shows that some high-scoring pages are not declining (false positives), while some declining pages receive lower scores (false negatives). The result is therefore treated as directional decision-support rather than proof that a refresh will cause improvement.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The baseline and models use the same held-out test rows and Precision@50 metric
- [x] Leakage check excludes `trend_direction` and `trend_pct`
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
